In [6]:
import jax 
import jax.numpy as jnp 
import gymnax
from flax import nnx

In [2]:
# obtain random keys for determinism

seed = 0
key = jax.random.key(seed=seed)
key, key_reset, key_policy, key_step = jax.random.split(key, 4)

In [3]:
# env and env_params are split because JAX requires 
# pure functions and immutable, explicit state.

# env_params is its own thing, which allows a user to use 
# multiple env_params over a vmap. 


env, env_params = gymnax.make(
    "Pendulum-v1"
)

# env_params = env_params.replace(dt=5), need to do this since env_params is immutable


In [8]:
obs, state = env.reset(key_reset, env_params)
action = env.action_space(env_params).sample(key_policy)

obs, state, reward, done, question = env.step(key_step, state, action, env_params)

print(obs.shape)
obs, state, reward, done, question, action

(3,)


(Array([-0.9999658, -0.0082727, -0.7511071], dtype=float32),
 EnvState(time=Array(1, dtype=int32, weak_type=True), theta=Array(-3.1333199, dtype=float32), theta_dot=Array(-0.7511071, dtype=float32), last_u=Array(1.609798, dtype=float32)),
 Array(-9.678166, dtype=float32),
 Array(False, dtype=bool, weak_type=True),
 {'discount': Array(1., dtype=float32, weak_type=True)},
 Array([1.609798], dtype=float32))

In [12]:
# can vmap; look at reset() and step() args to see how the vmap in_axes works

vmap_reset = jax.vmap(env.reset, in_axes=(0, None))
vmap_step = jax.vmap(env.step, in_axes=(0, 0, 0, None))
vmap_action = jax.vmap(env.action_space(env_params).sample, in_axes=0)  # sample only takes one arg so no need in_axes=(0)

num_envs = 8 

# splitting keys: num_envs * 4 
key, key_reset, key_policy, key_step  = jax.random.split(key, 4)
vmap_key_reset = jax.random.split(key_reset, num_envs)
vmap_key_policy = jax.random.split(key_policy, num_envs)
vmap_key_step = jax.random.split(key_step, num_envs)


obs, state = vmap_reset(vmap_key_reset, env_params)
action = vmap_action(vmap_key_policy)
obs, state, reward, done, gamma, = vmap_step(
    vmap_key_step, state, action, env_params
)

print("obs:", obs.shape)
print("action:", action.shape)

obs: (8, 3)
action: (8, 1)


In [1]:
from tppo.algorithms.base_tppo import TransformerPPO 
from tppo.utils.load_configs import load_config


cfg = load_config("example_model.toml")
print(cfg)
#tppo = TransformerPPO(T, d_hidden, d_keys, d_vals, d_ff, rngs, band, num_layers)


{'example_model': {'T': 100, 'd_hidden': 64, 'd_keys': 64, 'd_vals': 64, 'd_ff': 128, 'num_layers': 1, 'band': 100}}
